In [8]:
import numpy as np
import pandas as pd
from casadi import *
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *
from scipy.interpolate import CubicSpline
from sklearn.metrics import root_mean_squared_error as RMSE

def TurningSpeed(u_control_1, V_cruising, V_turning):
    """
    Calcula a velocidade de referência adaptativa com base no ângulo de esterçamento.
    
    Parâmetros:
    - u_control_1: float, ângulo atual das rodas em graus [-30, 30]
    - V_cruising: float, velocidade de cruzeiro em linha reta (ex: 20 m/s)
    - V_turning: float, velocidade reduzida limite para curvas acentuadas (ex: 8 m/s)
    
    Retorna:
    - V_ref: float, velocidade ótima calculada
    """
    # 1. Trabalha com o valor absoluto para garantir comportamento simétrico
    angulo_abs = abs(u_control_1)
    
    # 2. Define os pontos de quebra da interpolação baseados no módulo do ângulo:
    # De 0° até 3°: Retorna V_cruising
    # De 3° até 30°: Reduz proporcionalmente até V_turning
    pontos_angulo = np.linspace(0,21,2000)
    pontos_velocidade = np.flip(np.linspace(V_turning, V_cruising, 2000))
    
    # 3. Interpola linearmente de forma segura e blinda contra inputs fora do escopo
    V_ref = np.interp(angulo_abs, pontos_angulo, pontos_velocidade)
    
    return float(V_ref)

In [9]:
def SimulateRT(dt=0.1,n_horizon = 5,sim_steps = 3000,W_X = 1,W_Y = 1,W_speed = 5 ,W_acc = 1.5,W_delta = 0.25,W_U0 = 15,W_U1 = 2, show=False):

    BreakCheck = False
    V_cruising = 16
    V_max = 25
    V_turning = 8
    track_percentual = 1
    size = 2
    path = r'DyntheticDataset\RaceTrack4.csv'
    df = pd.read_csv(path)
    x_mid = df['x_coords'].values[:] * size
    y_mid = df['y_coords'].values[:] * size

    # Ensure the track loops closed cleanly
    if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
        x_mid = np.append(x_mid, x_mid[0])
        y_mid = np.append(y_mid, y_mid[0])

    track_points = np.vstack((x_mid, y_mid)).T
    track_tree = KDTree(track_points)

    # Calculate cumulative distance (arc length 's') along track coordinates
    dx = np.diff(x_mid)
    dy = np.diff(y_mid)
    segment_lengths = np.sqrt(dx**2 + dy**2)
    s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
    track_length = s_coor[-1]

    # Linear interpolators to find exact continuous (x, y) coordinates for any distance s
    get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
    get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

    l = 2.5 # Wheelbase

    opti = Opti()

    # Decision variables over the prediction horizon
    X = opti.variable(4, n_horizon + 1) # States: [x_pos, y_pos, v, psi]
    U = opti.variable(2, n_horizon)     # Controls: [a, delta]

    # Parameters
    X0 = opti.parameter(4)
    X_REF = opti.parameter(n_horizon)
    Y_REF = opti.parameter(n_horizon)
    V_REF = opti.parameter(n_horizon)
    U_PREV = opti.parameter(2)  

    obj = 0
    opti.subject_to(X[:, 0] == X0)

    # Build the execution graph symbolic tree once
    for k in range(n_horizon):
        x_k   = X[0, k]
        y_k   = X[1, k]
        v_k   = X[2, k]
        psi_k = X[3, k]
        
        a_k     = U[0, k]
        delta_k = U[1, k]

        beta = atan(0.5 * tan(delta_k))
        
        next_x   = x_k + dt * (v_k * cos(psi_k + beta))
        next_y   = y_k + dt * (v_k * sin(psi_k + beta))
        next_v   = v_k + dt * a_k
        next_psi = psi_k + dt * ((v_k / l) * sin(beta))

        opti.subject_to(X[0, k+1] == next_x)
        opti.subject_to(X[1, k+1] == next_y)
        opti.subject_to(X[2, k+1] == next_v)
        opti.subject_to(X[3, k+1] == next_psi)

        # Base state and input magnitudes stage cost
        obj += W_X*(x_k - X_REF[k])**2 + W_Y*(y_k - Y_REF[k])**2 + W_speed * (v_k - V_REF[k])**2
        obj += W_acc * a_k**2 + W_delta * delta_k**2

        # Slew rate penalties inside the preview horizon steps
        if k > 0:
            obj += W_U0 * (U[0, k] - U[0, k-1])**2  # Jerk penalty
            obj += W_U1 * (U[1, k] - U[1, k-1])**2   # Steering rate penalty

    # Smooth transitions linking external past inputs directly to the initial horizon decision step
    #obj += 25.0 * (U[0, 0] - U_PREV[0])**2
    #obj += 2.0 * (U[1, 0] - U_PREV[1])**2

    # Terminal cost step calculation
    #obj += (X[0, n_horizon] - X_REF[n_horizon - 1])**2 + \
    #       (X[1, n_horizon] - Y_REF[n_horizon - 1])**2 + \
    #       W_speed * (X[2, n_horizon] - V_REF[n_horizon - 1])**2

    opti.minimize(obj)

    # Operational Bound limits
    opti.subject_to(opti.bounded(-7.0, U[0, :], 3))                 
    opti.subject_to(opti.bounded(-np.deg2rad(30), U[1, :], np.deg2rad(30))) 
    opti.subject_to(opti.bounded(0.0, X[2, :], V_cruising*1))                  

    # Setup solver configuration options once
    opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.warm_start_init_point": "yes"}
    opti.solver('ipopt', opts)


    # %%
    # %% [markdown]
    # 3. CLOSED-LOOP SIMULATION LOOP

    # %%
    s_total_traveled = 0.0
    last_current_idx = 0

    x0_start = np.array([100, y_mid[0], 0.0, 0.0])
    x_current = x0_start.copy()
    current_time = 0.0

    # Memory array tracking previous system inputs (Starts at 0.0 standstill condition)
    u_executed_prev = np.array([0.0, 0.0])

    t_history = [current_time]
    x_history = [x_current[0]]
    y_history = [x_current[1]]
    xRef_history = [x_current[0]]
    yRef_history = [x_current[1]]
    v_history = [x_current[2]]
    v_ref_history = [0]
    v_k_history = []
    v_horizon = []
    acc_history = [0]
    delta_history = [0]
    psi_history = [0]
    is_turning_sharp = False
    turning = [0]

    # Continuous trajectory memory for warm-starting
    last_sol_X = np.tile(x_current.reshape(4, 1), (1, n_horizon + 1))
    last_sol_U = np.zeros((2, n_horizon))

    for step in range(sim_steps):
        if (step) % 300 == 0 and show: 
            print('step:', step, s_total_traveled, track_length,x_current[2])
            
        _, current_idx = track_tree.query([x_current[0], x_current[1]])
        
        idx_diff = current_idx - last_current_idx
        if idx_diff < -len(x_mid)/2: 
            idx_diff += len(x_mid)
        elif idx_diff > len(x_mid)/2:
            idx_diff -= len(x_mid)
            
        if idx_diff > 0:
            s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
        last_current_idx = current_idx

        # Generate Time-Varying Horizon Reference Profiles
        s_projected = s_coor[current_idx]
        x_ref_horizon = np.zeros(n_horizon)
        y_ref_horizon = np.zeros(n_horizon)
        v_ref_horizon = np.zeros(n_horizon)

        for k in range(n_horizon):
            s_projected += max(x_current[2], 1.5) * dt 
            s_wrapped = s_projected % track_length
            
            x_ref_horizon[k] = get_x_at_s(s_wrapped)
            y_ref_horizon[k] = get_y_at_s(s_wrapped)

            if is_turning_sharp:
                if s_total_traveled < track_length:
                    speed = TurningSpeed(np.rad2deg(u_control[1]),V_cruising, V_turning)
                    v_ref_horizon[k] = speed
                elif s_total_traveled >= track_length:
                    v_ref_horizon[k] = 0

            elif s_total_traveled >= track_length * track_percentual:
                v_ref_horizon[k] = 0.0

            else:
                v_ref_horizon[k] = V_cruising
        # Update parameters mapping arrays
        opti.set_value(X0, x_current)
        opti.set_value(X_REF, x_ref_horizon)
        opti.set_value(Y_REF, y_ref_horizon)
        opti.set_value(V_REF, v_ref_horizon)
        opti.set_value(U_PREV, u_executed_prev)  # Push historical tracking input directly to solver memory

        # Seed guesses
        opti.set_initial(X, last_sol_X)
        opti.set_initial(U, last_sol_U)

        try:
            sol = opti.solve()
            u_control = sol.value(U[:, 0])
            last_sol_X = sol.value(X)
            last_sol_U = sol.value(U)
        except Exception as e:
            # Fallback sub-optimal state capture strategy
            u_control = opti.debug.value(U[:, 0])
            last_sol_X = opti.debug.value(X)
            last_sol_U = opti.debug.value(U)

        # Save the selected control inputs to match with the next tracking frame interval
        u_executed_prev = u_control.copy()

        # --- Plant Simulator (Process Step using Forward Euler) ---
        beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
        x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
        y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
        v_next = x_current[2] + dt * u_control[0]
        psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

        is_turning_sharp = (u_control[1] >= np.deg2rad(3)) or (u_control[1] <= -np.deg2rad(3))
        turning.append(is_turning_sharp)
        
        x_current = np.array([x_next, y_next, v_next, psi_next])
        current_time += dt

        t_history.append(current_time)
        x_history.append(x_current[0])
        y_history.append(x_current[1])
        xRef_history.append(x_ref_horizon[0])
        yRef_history.append(y_ref_horizon[0])
        v_history.append(x_current[2])
        v_k_history.append(last_sol_X[2])
        v_ref_history.append(v_ref_horizon[0])
        acc_history.append(u_control[0])
        psi_history.append(x_current[3])
        delta_history.append(u_control[1])
        v_horizon.append(v_ref_horizon)

        delta_check = np.rad2deg(np.abs(np.array(delta_history)))
        delta_check = delta_check[delta_check>26]

        if s_total_traveled >= track_length and x_current[2] <= 0.99:
            if show:
                print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
            break
        elif s_total_traveled >= 1.1*track_length:
            if show:
                print('step:', step, s_total_traveled, track_length,x_current[2])
            BreakCheck = True
            break

        elif len(delta_check) > 0:
            BreakCheck = True
            break

    # Convert historical data into numpy arrays for rendering 
    t_history = np.array(t_history)
    x_history = np.array(x_history)
    y_history = np.array(y_history)
    xRef_history = np.array(xRef_history)
    yRef_history = np.array(yRef_history)
    v_history = np.array(v_history)
    v_ref_history = np.array(v_ref_history)
    acc_history = np.array(acc_history)
    delta_history = np.array(delta_history)
    delta_history = np.rad2deg(delta_history)

    psi_history = np.array(psi_history)
    psi_history = np.rad2deg(psi_history)
    turning = [0 if t==False else 1 for t in turning]
    turning = np.array(turning)

    score = RMSE(v_ref_history, v_history)
    ySeries=[v_ref_history, v_history, acc_history, delta_history, turning]
    xSeries=[t_history for i in range(len(ySeries))]
    names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
            'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
    data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]

    return score,BreakCheck,[xSeries,ySeries,names,data]

#score,[xSeries,ySeries,names,data] = SimulateRT(n_horizon=30,W_X = 1,W_Y = 1,W_speed = 5,W_acc = 1.5,W_delta = 0.25,W_U0 = 15,W_U1 = 2)

In [21]:
import optuna
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.samplers import RandomSampler
from optuna.exceptions import TrialPruned

def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        # Mantendo multivariate=True para capturar a correlação entre os pesos do MPC
        # Ajustado n_startup_trials=100 para condizer bem com seus 500 n_trials totais
        sampler = optuna.samplers.TPESampler(
            multivariate=True, 
            n_startup_trials=100
        )
    elif mode == 'random':
        sampler = RandomSampler()
    return sampler

def objective(trial):

    W_X = 1.0
    W_Y = 1.0

    #W_X = trial.suggest_float('W_X', 1, 1)
    #W_Y = trial.suggest_float('W_Y', 1, 1)
    W_speed = trial.suggest_float('W_speed', 1e-2, 30,step=1e-2)
    W_acc = trial.suggest_float('W_acc'    , 1e-2, 30,step=1e-2)
    W_delta = trial.suggest_float('W_delta', 1e-2, 30,step=1e-2)
    W_U0 = trial.suggest_float('W_U0'      , 1e-2, 30,step=1e-2)
    W_U1 = trial.suggest_float('W_U1'      , 1e-2, 30,step=1e-2)
    n_horizon=30
    dt = 0.1
    sim_steps=800
    score,BreakCheck,_ = SimulateRT(dt,n_horizon,sim_steps,
                                W_X,W_Y,W_speed,W_acc,W_delta,W_U0,W_U1)
    if BreakCheck:
        raise TrialPruned()
    
    return score

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction='minimize',
    #directions=["minimize", "minimize"],
    sampler=SelSampler(mode='tpe'),
    #pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)

study.optimize(objective, n_trials=500)
best_params = study.best_params
params = list(best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)


c:\Users\Claudio\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-06-25 09:46:16,878] A new study created in memory with name: no-name-b84fd853-63fc-4f20-927d-65bd33607eac
[I 2026-06-25 09:46:17,545] Trial 0 pruned. 
[I 2026-06-25 09:46:18,215] Trial 1 pruned. 
[I 2026-06-25 09:46:18,420] Trial 2 pruned. 
[I 2026-06-25 09:46:23,898] Trial 3 pruned. 
[I 2026-06-25 09:46:24,115] Trial 4 pruned. 
[I 2026-06-25 09:46:24,291] Trial 5 pruned. 
[I 2026-06-25 09:46:24,547] Trial 6 pruned. 
[I 2026-06-25 09:46:24,974] Trial 7 pruned. 
[I 2026-06-25 09:46:25,354] Trial 8 pruned. 
[I 2026-06-25 09:46:25,993] Trial 9 pruned. 
[I 2026-06-25 09:46:33,315] Trial 10 pruned. 
[I 2026-06-25 09:46:35,701] Trial 11 pruned. 
[I 2026-06-25 09:46:36,289] Trial 12 pruned. 
[I 2026-06-25 09:46:36,895] Trial 13 pruned. 
[I 2026-06-25 09:46

Erro: 3.1366020882495698 parameters:  [10.98, 20.610000000000003, 15.67, 28.46, 20.360000000000003]


In [22]:
W_speed,W_acc,W_delta,W_U0,W_U1 = params
W_X = 1.0
W_Y = 1.0
dt=0.1
n_horizon=30
sim_steps = 1000
score,BreakCheck,[xSeries,ySeries,names,data] = SimulateRT(dt,n_horizon,sim_steps,
                                                W_X,W_Y,W_speed,W_acc,W_delta,W_U0,W_U1,show=True)
PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title='Parameters Over Time')
PlotMPCTracksPLY(data,width=800,height=500)

step: 0 0.0 748.2944182245575 0.0
step: 300 383.5193782638242 748.2944182245575 11.801646041267974
Simulation completed! Lap finished and vehicle stopped on centerline at step 595.


In [12]:
data[5] = np.abs(data[5])
print(len(data[5][data[5]>28]))
PlotMPCTracksPLY(data,width=800,height=500)

0
